In [1]:
!pip install -q transformers==4.40.2
!pip install -q ranx
!pip install -q pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.6 MB/s eta 

In [2]:
# Top 1k candidate docs
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz
!gunzip msmarco-passagetest2019-top1000.tsv.gz

# 43 queries
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
!gunzip msmarco-test2019-queries.tsv.gz

# Qrels
!wget https://trec.nist.gov/data/deep/2019qrels-pass.txt

--2025-05-25 18:02:24--  https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634062 (25M) [application/x-gzip]
Saving to: ‘msmarco-passagetest2019-top1000.tsv.gz’

msmarco-passagetest 100%[===================>]  25.40M  5.76MB/s    in 6.4s    

2025-05-25 18:02:31 (4.00 MB/s) - ‘msmarco-passagetest2019-top1000.tsv.gz’ saved [26634062/26634062]

--2025-05-25 18:02:32--  https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting 

In [3]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    AutoTokenizer, AutoModelForSequenceClassification
)
from ranx import Run, Qrels, evaluate

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


queries_df = pd.read_csv("msmarco-test2019-queries.tsv", sep="\t", header=None, names=["query_id", "query"])
candidates_df = pd.read_csv("msmarco-passagetest2019-top1000.tsv", sep="\t", header=None,
                            names=["query_id", "passage_id", "query_text", "passage_text"])
qrels_df = pd.read_csv("2019qrels-pass.txt", sep=r"\s+", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
qrels_df.to_csv("2019qrels-fixed.txt", sep="\t", index=False, header=False)
qrels = Qrels.from_file("2019qrels-fixed.txt", kind="trec")

query_dict_original = dict(zip(queries_df["query_id"].astype(str), queries_df["query"]))

In [5]:
models = {
    "minilm": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "tinybert": "cross-encoder/ms-marco-TinyBERT-L-2-v2"
}

In [6]:
def queries_expansion(query_dict):

    tokenizer = T5Tokenizer.from_pretrained("t5-base")
    model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)
    expanded_queries = {}

    for qid, query in tqdm(query_dict.items(), desc="Expanding queries"):
        prompt = f"Expand this query: {query}"
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            output = model.generate(input_ids, max_length=100, do_sample=True, top_p=0.9)
        expanded_query = tokenizer.decode(output[0], skip_special_tokens=True)
        expanded_queries[qid] = expanded_query.strip()

    return expanded_queries


In [7]:
def rerank_and_save(query_dict, query_type):

    for model_name, model_path in models.items():
        print(f"\nReranking {query_type} being done for the queries with {model_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device).eval()
        output_lines = []

        for qid, query in tqdm(query_dict.items(), desc=f"Processing {model_name}"):
            docs = candidates_df[candidates_df["query_id"] == int(qid)]
            scores = []
            for _, row in docs.iterrows():
                inputs = tokenizer(
                    query,
                    str(row["passage_text"]),
                    truncation=True,
                    return_tensors="pt",
                    max_length=512
                ).to(device)
                with torch.no_grad():
                    score = model(**inputs).logits[0].item()
                scores.append((str(row["passage_id"]), score))

            # Sort by score and format TREC run
            ranked = sorted(scores, key=lambda x: x[1], reverse=True)
            for rank, (doc_id, score) in enumerate(ranked):
                output_lines.append(f"{qid} Q0 {doc_id} {rank+1} {score:.4f} {query_type}-{model_name}")

        # Save run file
        run_file = f"ranking_{query_type}_{model_name}.run"
        with open(run_file, "w") as f:
            f.write("\n".join(output_lines) + "\n")
        print(f" Saving the : {run_file}")

In [8]:
expanded_queries = queries_expansion(query_dict_original)
with open("expanded_queries.json", "w") as f:
    json.dump(expanded_queries, f, indent=2)

rerank_and_save(query_dict_original, "original")
rerank_and_save(expanded_queries, "expanded")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

... (21 outputs left out)

Processing minilm: 100%|██████████| 200/200 [18:56<00:00,  5.68s/it]


 Saving the : ranking_expanded_minilm.run

Reranking expanded being done for the queries with tinybert...


Processing tinybert: 100%|██████████| 200/200 [09:22<00:00,  2.81s/it]

 Saving the : ranking_expanded_tinybert.run


In [9]:
def topdocs_getting(run_file, top_k=3):

    top_docs = {}
    with open(run_file, "r") as f:
        for line in f:
            qid, _, doc_id, _, _, _ = line.strip().split()
            if qid not in top_docs:
                top_docs[qid] = []
            if len(top_docs[qid]) < top_k:
                passage = candidates_df[candidates_df["passage_id"] == int(doc_id)]["passage_text"].values
                if len(passage) > 0:
                    top_docs[qid].append(passage[0])
    return top_docs

In [10]:
def expand_queries_prf(query_dict, top_docs_dict):

    tokenizer = T5Tokenizer.from_pretrained("t5-base")
    model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)
    expanded_queries_prf = {}

    for qid in tqdm(query_dict.keys(), desc="PRF Expansion"):
        if qid not in top_docs_dict:
            continue
        prompt = (
            f"Reformulation should be done for the queries based on the relevant documents:\n"
            f"Query: {query_dict[qid]}\n"
            f"Documents:\n" + "\n".join(f"[Doc {i+1}] {doc}" for i, doc in enumerate(top_docs_dict[qid])) +
            f"\nExpanded Query:"
        )
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            output = model.generate(input_ids, max_length=128)
        expanded_queries_prf[qid] = tokenizer.decode(output[0], skip_special_tokens=True).strip()

    return expanded_queries_prf

In [11]:
for model_name in models:
    print(f"\n=== Generating PRF expansions for {model_name} ===")
    top_docs_dict = topdocs_getting(f"ranking_original_{model_name}.run")
    expanded_queries_prf = expand_queries_prf(query_dict_original, top_docs_dict)

    # Save PRF expansions
    prf_file = f"expanded_queries_prf_{model_name}.json"
    with open(prf_file, "w") as f:
        json.dump(expanded_queries_prf, f, indent=2)
    print(f" Saving the : {prf_file}")

    # Rerank with PRF expansions
    rerank_and_save(expanded_queries_prf, f"expanded_prf_{model_name}")



=== Generating PRF expansions for minilm ===


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
PRF Expansion: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s]


 Saving the : expanded_queries_prf_minilm.json

Reranking expanded_prf_minilm being done for the queries with minilm...


Processing minilm: 100%|██████████| 200/200 [19:35<00:00,  5.88s/it]


 Saving the : ranking_expanded_prf_minilm_minilm.run

Reranking expanded_prf_minilm being done for the queries with tinybert...


Processing tinybert: 100%|██████████| 200/200 [09:41<00:00,  2.91s/it]


... (3 outputs left out)

Processing minilm: 100%|██████████| 200/200 [19:15<00:00,  5.78s/it]


 Saving the : ranking_expanded_prf_tinybert_minilm.run

Reranking expanded_prf_tinybert being done for the queries with tinybert...


Processing tinybert: 100%|██████████| 200/200 [09:42<00:00,  2.91s/it]

 Saving the : ranking_expanded_prf_tinybert_tinybert.run


In [14]:
from ranx import Run, Qrels, evaluate
import pandas as pd

qrels = Qrels.from_file("2019qrels-fixed.txt", kind="trec")

results = []

# Define run file mappings
run_files = {
    "minilm": {
        "original": "ranking_original_minilm.run",
        "expanded": "ranking_expanded_minilm.run",
        "prf": "ranking_expanded_prf_minilm_minilm.run"
    },
    "tinybert": {
        "original": "ranking_original_tinybert.run",
        "expanded": "ranking_expanded_tinybert.run",
        "prf": "ranking_expanded_prf_tinybert_tinybert.run"
    }
}

metrics = ["ndcg@10", "recall@100", "map@1000"]

for model_name, paths in run_files.items():
    run_orig = Run.from_file(paths["original"], kind="trec")
    run_exp = Run.from_file(paths["expanded"], kind="trec")
    run_prf = Run.from_file(paths["prf"], kind="trec")

    res_orig = evaluate(qrels, run_orig, metrics, make_comparable=True)
    res_exp = evaluate(qrels, run_exp, metrics, make_comparable=True)
    res_prf = evaluate(qrels, run_prf, metrics, make_comparable=True)

    results.extend([
        {"Model": model_name, "Query Type": "Original", **res_orig},
        {"Model": model_name, "Query Type": "Standard Expansion", **res_exp},
        {"Model": model_name, "Query Type": "PRF Expansion", **res_prf}
    ])



/usr/local/lib/python3.11/dist-packages/ranx/metrics/ndcg.py:72: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _ndcg(qrels[i], run[i], k, rel_lvl, jarvelin)


In [15]:
print("\n === Evaluation results comparision====")
results_prf_df = pd.DataFrame(results)
print(results_prf_df)


 === Evaluation results comparision====
      Model          Query Type   ndcg@10  recall@100  map@1000
0    minilm            Original  0.735793    0.533204  0.498550
1    minilm  Standard Expansion  0.269896    0.260053  0.216093
2    minilm       PRF Expansion  0.586349    0.439014  0.400862
3  tinybert            Original  0.698411    0.512776  0.461743
4  tinybert  Standard Expansion  0.270936    0.252736  0.212091
5  tinybert       PRF Expansion  0.571557    0.424202  0.372034
